## Split Overmerged Authors — Phase 1: Zero-Overlap Last Names

Identifies work_authors where the parsed last name of `raw_author_name` has ZERO overlap with the author profile's parsed last name. These are clearly wrong matches (different people) that need to be split by nulling `author_id`.

**Safety filters:**
- Excludes East Asian first/last swaps (Phase 2)
- Excludes work_authors where the work's raw_orcid (from crossref/datacite) matches the author profile's ORCID
- Requires both last names > 2 chars (excludes single-char or empty)
- No substring overlap in either direction

### Cell 1: Build candidates table

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.overmerge_split_candidates AS
WITH work_parsed AS (
  SELECT wa.work_id, wa.author_sequence, wa.author_id, wa.raw_author_name,
    an_w.parsed_name.last AS work_last,
    an_w.parsed_name.first AS work_first
  FROM openalex.works.work_authors wa
  JOIN openalex.authors.author_names an_w ON wa.raw_author_name = an_w.raw_author_name
  WHERE wa.author_id IS NOT NULL
    AND an_w.parsed_name.last IS NOT NULL
    AND LENGTH(an_w.parsed_name.last) > 1
),
author_parsed AS (
  SELECT oa.id AS author_id,
    an_a.parsed_name.last AS author_last,
    an_a.parsed_name.first AS author_first,
    oa.orcid, oa.works_count
  FROM openalex.authors.openalex_authors oa
  JOIN openalex.authors.author_names an_a ON oa.full_name = an_a.raw_author_name
),
-- Get raw_orcid from source metadata per work+author_sequence
work_orcids AS (
  SELECT wb.id AS work_id, pos AS author_sequence, authorship.raw_orcid
  FROM openalex.works.openalex_works_base wb
  LATERAL VIEW POSEXPLODE(wb.authorships) t AS pos, authorship
  WHERE authorship.raw_orcid IS NOT NULL AND authorship.raw_orcid != ''
)
SELECT wp.work_id, wp.author_sequence, wp.author_id, wp.raw_author_name,
  wp.work_last, wp.work_first, ap.author_last, ap.author_first,
  ap.orcid, ap.works_count
FROM work_parsed wp
JOIN author_parsed ap ON wp.author_id = ap.author_id
WHERE wp.work_last != ap.author_last
  AND LENGTH(wp.work_last) > 2 AND LENGTH(ap.author_last) > 2
  -- No substring overlap in either direction
  AND INSTR(wp.work_last, ap.author_last) = 0
  AND INSTR(ap.author_last, wp.work_last) = 0
  -- Exclude East Asian first/last swaps (Phase 2)
  AND NOT (wp.work_last = ap.author_first AND wp.work_first = ap.author_last)
  -- Exclude only where the work's raw_orcid confirms this author assignment
  AND NOT EXISTS (
    SELECT 1 FROM work_orcids wo
    WHERE wo.work_id = wp.work_id
      AND wo.author_sequence = wp.author_sequence
      AND wo.raw_orcid = ap.orcid
  )

### Cell 2: Validation statistics

In [ ]:
SELECT COUNT(*) AS total_candidates,
  COUNT(DISTINCT author_id) AS distinct_authors,
  COUNT(DISTINCT work_id) AS distinct_works,
  PERCENTILE_APPROX(works_count, ARRAY(0.25, 0.5, 0.75, 0.95)) AS author_works_pctiles
FROM openalex.authors.overmerge_split_candidates

### Cell 3: Spot-check sample (manual review)

In [ ]:
SELECT work_id, author_id, raw_author_name,
  work_last, work_first,
  author_last, author_first,
  works_count AS author_works_count
FROM openalex.authors.overmerge_split_candidates
ORDER BY RAND()
LIMIT 50

### Cell 4: Create audit log (for rollback)

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.overmerge_split_log_phase1 AS
SELECT work_id, author_sequence, author_id AS previous_author_id,
  raw_author_name, work_last, author_last,
  current_timestamp() AS split_timestamp
FROM openalex.authors.overmerge_split_candidates

### Cell 5: Execute the split

**WARNING**: This nulls out author_ids. Verify cells 2-3 before running.

**NOTE**: MatchAuthors has cutoffs (`work_id > 7B`, `created_date >= 2025-12-20`). Older split records will NOT be re-processed automatically. A separate re-matching run or cutoff relaxation is needed.

In [ ]:
MERGE INTO openalex.works.work_authors AS target
USING openalex.authors.overmerge_split_candidates AS source
ON target.work_id = source.work_id
   AND target.author_sequence = source.author_sequence
WHEN MATCHED THEN
  UPDATE SET
    target.author_id = NULL,
    target.updated_at = current_timestamp()

### Cell 6: Post-split verification

In [ ]:
SELECT
  COUNT(*) AS nulled_records,
  COUNT(DISTINCT c.work_id) AS works_affected
FROM openalex.works.work_authors wa
JOIN openalex.authors.overmerge_split_log_phase1 c
  ON wa.work_id = c.work_id AND wa.author_sequence = c.author_sequence
WHERE wa.author_id IS NULL

### Cell 7: Outcome tracking (run after MatchAuthors re-processes)

In [ ]:
SELECT
  CASE
    WHEN wa.author_id IS NULL THEN 'STILL_UNMATCHED'
    WHEN wa.author_id = log.previous_author_id THEN 'RE_MATCHED_SAME'
    ELSE 'RE_MATCHED_NEW'
  END AS outcome,
  COUNT(*) AS cnt
FROM openalex.works.work_authors wa
JOIN openalex.authors.overmerge_split_log_phase1 log
  ON wa.work_id = log.work_id AND wa.author_sequence = log.author_sequence
GROUP BY 1